Imports and paths

In [159]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

if not DATA_RAW_PATH.exists():
    raise FileExistsError()

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: PosixPath('/Users/alainkhan/Dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: PosixPath('/Users/alainkhan/Dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: PosixPath('/Users/alainkhan/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


PosixPath('/Users/alainkhan/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

In [160]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dfs: dict[str, DataFrame] = {f.name: pd.read_csv(f, dtype=str) for f in raw_filepaths}
# ic(dfs)

In [161]:
A = ""
len(A)

0

In [162]:
def lev_d(a: str, b: str) -> int:
    """Levenshtein distance
    O(mn)
    """
    lena = len(a)
    lenb = len(b)

    if lenb == 0:
        return lena

    if lena == 0:
        return lenb

    heada = a[0]
    headb = b[0]
    taila = a[1:]
    tailb = b[1:]

    if heada == headb:
        return lev_d(taila, tailb)

    return 1 + min(lev_d(taila, b), lev_d(a, tailb), lev_d(taila, tailb))


def sim_norm_lev_d(a: str, b: str) -> float:
    """similarity normalised lev_d"""
    lena = len(a)
    lenb = len(b)

    return 1 - lev_d(a, b) / max(lena, lenb)


def dice_sim(A: set[str], B: set[str]) -> float:
    """Dice similarity
    |AnB| / (|A|+|B|/2)
    """
    return 2 * len(A.intersection(B)) / (len(A) + len(B))

In [163]:
import string

A = "0123456789abcdefABCDEFg"

p = set(string.hexdigits)
q = set(A)

print(1 - len(p - q) / max(len(p), len(q)))

1.0


In [170]:
import string


def infer_dtypes(df: DataFrame) -> DataFrame:
    """
    df -> out_df

    ascii: 0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~
    if any entry in col is not in ascii, then use national ('n' in 'nvarchar')

    df rows start at 0
    """

    def is_chars_in_string(chars: str, parent_string: str) -> bool:
        if set(chars).intersection(set(parent_string)):
            return True

        return False

    out_df = pd.DataFrame(
        index=df.columns,
        columns=[
            "has_nulls",
            "where_nulls",
            "total_nulls",
            "sorted_chars_used",
            "unique_chars_used",
            "has_digits",
            "dice_sim_to_digits",
            "has_hex_digits",
            "dice_sim_to_hex_digits",
            "has_decimal",
            "has_dash",
            "has_colon",
            "has_space",
            "has_ascii",
            "dice_sim_to_ascii",
            "has_non_ascii",
            "dice_sim_to_non_ascii",
            "has_prefix_zero",
            "entry_lengths",
            "unique_entry_lengths",
            "max_entry_length",
            "is_fixed_length",
            "has_unique_entries",
            "has_exactly_two_entries",
        ],
    )

    cols = df.columns

    for col in cols:
        print(col)

        where_null = df[col].isnull()

        out_df.loc[col, "where_nulls"] = df[where_null].index.tolist()
        out_df.loc[col, "has_nulls"] = 1 if where_null.any() else 0
        out_df.loc[col, "total_nulls"] = where_null.sum()

        clean_series = df[col].dropna()

        chars_used: set[str] = set("".join(clean_series.astype(str)))
        sorted_chars_used: str = "".join(sorted(chars_used))
        out_df.loc[col, "sorted_chars_used"] = sorted_chars_used
        out_df.loc[col, "unique_chars_used"] = len(sorted_chars_used)

        out_df.loc[col, "has_digits"] = (
            1 if is_chars_in_string(string.digits, sorted_chars_used) else 0
        )
        out_df.loc[col, "has_hex_digits"] = (
            1 if is_chars_in_string(string.hexdigits, sorted_chars_used) else 0
        )
        out_df.loc[col, "has_decimal"] = (
            1 if is_chars_in_string(".", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_dash"] = (
            1 if is_chars_in_string("-", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_colon"] = (
            1 if is_chars_in_string(":", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_space"] = (
            1 if is_chars_in_string(" ", sorted_chars_used) else 0
        )
        out_df.loc[col, "has_ascii"] = (
            1 if is_chars_in_string(string.printable, sorted_chars_used) else 0
        )

        # if col chars > ascii when col chars - ascii > 0
        set_minus: set[str] = chars_used - set(string.printable)
        out_df.loc[col, "has_non_ascii"] = 1 if set_minus else 0

        out_df.loc[col, "dice_sim_to_digits"] = dice_sim(set(string.digits), chars_used)
        out_df.loc[col, "dice_sim_to_hex_digits"] = dice_sim(
            set(string.hexdigits), chars_used
        )
        out_df.loc[col, "dice_sim_to_ascii"] = dice_sim(
            set(string.printable), chars_used
        )
        out_df.loc[col, "dice_sim_to_non_ascii"] = dice_sim(set_minus, chars_used)

        unique_entries = clean_series.unique()

        out_df.loc[col, "has_prefix_zero"] = (
            1 if clean_series.astype(str).str.startswith("0").any() else 0
        )
        entry_lengths = sorted(clean_series.astype(str).str.len().unique().tolist())

        out_df.loc[col, "entry_lengths"] = entry_lengths
        out_df.loc[col, "unique_entry_lengths"] = len(entry_lengths)
        out_df.loc[col, "max_entry_length"] = max(entry_lengths)
        out_df.loc[col, "is_fixed_length"] = 1 if len(entry_lengths) == 1 else 0
        out_df.loc[col, "has_unique_entries"] = 1 if clean_series.is_unique else 0
        out_df.loc[col, "has_exactly_two_entries"] = (
            1 if len(unique_entries) == 2 else 0
        )

    return out_df

pd.set_option('display.max_columns', None)

filename = raw_filenames[6]
filename = "olist_" + "products" + "_dataset.csv"
print(filename)

df = dfs[filename]
infer_dtypes(df)

olist_products_dataset.csv
product_id
product_category_name
product_name_lenght
product_description_lenght
product_photos_qty
product_weight_g
product_length_cm
product_height_cm
product_width_cm


,has_nulls,where_nulls,total_nulls,sorted_chars_used,unique_chars_used,has_digits,dice_sim_to_digits,has_hex_digits,dice_sim_to_hex_digits,has_decimal,has_dash,has_colon,has_space,has_ascii,dice_sim_to_ascii,has_non_ascii,dice_sim_to_non_ascii,has_prefix_zero,entry_lengths,unique_entry_lengths,max_entry_length,is_fixed_length,has_unique_entries,has_exactly_two_entries
product_id,0,[],0,0123456789abcdef,16,1,0.769231,1,0.842105,0,0,0,0,1,0.275862,0,0.0,1,[32],1,32,1,1,0
product_category_name,1,"[105, 128, 145, 154, 197, 244, 294, 299, 347, ...",610,2_abcdefghijklmnopqrstuvwxyz,28,1,0.052632,1,0.28,0,0,0,0,1,0.4375,0,0.0,0,"[3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,...",28,46,0,0,0
product_name_lenght,1,"[105, 128, 145, 154, 197, 244, 294, 299, 347, ...",610,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2]",2,2,0,0,0
product_description_lenght,1,"[105, 128, 145, 154, 197, 244, 294, 299, 347, ...",610,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2, 3, 4]",4,4,0,0,0
product_photos_qty,1,"[105, 128, 145, 154, 197, 244, 294, 299, 347, ...",610,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2]",2,2,0,0,0
product_weight_g,1,"[8578, 18851]",2,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,1,"[1, 2, 3, 4, 5]",5,5,0,0,0
product_length_cm,1,"[8578, 18851]",2,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2, 3]",3,3,0,0,0
product_height_cm,1,"[8578, 18851]",2,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2, 3]",3,3,0,0,0
product_width_cm,1,"[8578, 18851]",2,0123456789,10,1,1.0,1,0.625,0,0,0,0,1,0.181818,0,0.0,0,"[1, 2, 3]",3,3,0,0,0


In [165]:
example_hex = "01223456789abcd234"
print(1 - len(set(string.hexdigits) - set(example_hex)) / len(set(string.hexdigits)))

0.6363636363636364


In [166]:
# scratch work

chars = "ab0z"
parent_string = string.ascii_letters

print(set(chars))
print(set(parent_string))


# print(dfs)
# print(dfs["olist_order_reviews_dataset.csv"]['review_comment_title'].unique().tolist())

{'a', 'b', '0', 'z'}
{'Z', 's', 'A', 'M', 't', 'V', 'k', 'Q', 'a', 'U', 'e', 'd', 'z', 'D', 'u', 'F', 'H', 'R', 'p', 'o', 'c', 'g', 'S', 'w', 'C', 'b', 'y', 'J', 'K', 'f', 'X', 'v', 'h', 'P', 'L', 'q', 'N', 'G', 'i', 'E', 'I', 'O', 'B', 'n', 'W', 'm', 'x', 'Y', 'j', 'r', 'l', 'T'}


In [167]:
(
    "has_nulls",
    "where_nulls",
    "total_nulls",
    "sorted_chars_used",
    "unique_chars_used",
    "has_digits",
    "dice_sim_to_digits",
    "has_hex_digits",
    "dice_sim_to_hex_digits",
    "has_decimal",
    "has_dash",
    "has_colon",
    "has_space",
    "has_ascii",
    "dice_sim_to_ascii",
    "has_non_ascii",
    "dice_sim_to_non_ascii",
    "has_prefix_zero",
    "entry_lengths",
    "unique_entry_lengths",
    "max_entry_length",
    "is_fixed_length",
    "has_unique_entries",
    "has_exactly_two_entries",
)

('has_nulls',
 'where_nulls',
 'total_nulls',
 'sorted_chars_used',
 'unique_chars_used',
 'has_digits',
 'dice_sim_to_digits',
 'has_hex_digits',
 'dice_sim_to_hex_digits',
 'has_decimal',
 'has_dash',
 'has_colon',
 'has_space',
 'has_ascii',
 'dice_sim_to_ascii',
 'has_non_ascii',
 'dice_sim_to_non_ascii',
 'has_prefix_zero',
 'entry_lengths',
 'unique_entry_lengths',
 'max_entry_length',
 'is_fixed_length',
 'has_unique_entries',
 'has_exactly_two_entries')

In [168]:
months = {
    "january",
    "february",
    "march",
    "april",
    "may",
    "june",
    "july",
    "august",
    "september",
    "october",
    "november",
    "december",
}

print(sorted(set("".join(string.ascii_lowercase)).difference(set("".join(months)))))

['k', 'q', 'w', 'x', 'z']


In [169]:
# dfs.keys()


# def describe_table(current_table: str, current_column: str) -> None:
#     df = dfs[current_table]

#     print(f"current column: {current_column}")
#     print("")
#     print(df[current_column].sample(5))
#     print("")
#     print(f"dtype:  {df[current_column].dtype}")
#     print(f"table:  {current_table}")
#     print(f"column: {current_column}")
#     print("")

#     # check if contains null
#     has_null = df[current_column].isnull().any()
#     print(f"contains NULL:  {has_null}")

#     # if has_null:
#     #     # filter out null
#     #     print("contains null")
#     #     df_notnull = df[df[current_column].notnull()]
#     #     pass

#     if df[current_column].dtype != "object":
#         print("not an object")

#         df[current_column] = df[current_column].dropna()

#         print(df[current_column].describe())

#         return

#     # check if requires 'n' prefix: nchar, nvarchar
#     # n = national, means contains unicode
#     is_national: bool = not df[current_column].apply(lambda x: str(x).isascii()).all()

#     # check if char or varchar
#     entry_lengths = df[current_column].dropna().str.len().unique()

#     max_length = entry_lengths.max()
#     is_fixed_length = entry_lengths.size == 1

#     # print(f'is char:        {is_char}')
#     print(f"is above ascii: {is_national}")
#     print(f"entry lengths:  {entry_lengths}")
#     print(f"max length:     {max_length}")
#     print(f"fixed length:   {is_fixed_length}")

# table = "olist_geolocation_dataset.csv"
# column = "geolocation_state"
# describe_table(current_table=table, current_column=column)